# TFG · Lección 1 (sistemas individuales): comprobaciones en Qiskit

Cada bloque comprueba numéricamente un resultado de la Sección 3.1 del cuaderno.
Ejecuta las celdas **en orden**. Las salidas y figuras que obtengas son las que van al documento.

In [ ]:
# Versión de Qiskit (apúntala en el cuaderno)
from qiskit import __version__
print(__version__)

In [ ]:
import numpy as np
from numpy import sqrt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram

## Experimento 1 · Estados cuánticos válidos y no válidos

Un vector es un estado cuántico si y solo si tiene norma euclídea 1.
`w = (1/3, 2/3)` tiene entradas no negativas pero no es un estado: su norma es $\sqrt{5}/3 \approx 0.745$.

In [ ]:
mas = Statevector([1 / sqrt(2), 1 / sqrt(2)])      # |+>
psi = Statevector([(1 + 2j) / 3, -2 / 3])           # ejemplo del cuaderno
w   = Statevector([1 / 3, 2 / 3])                   # NO es un estado

for nombre, v in [("|+>", mas), ("psi", psi), ("w", w)]:
    print(f"{nombre:4s}  norma = {np.linalg.norm(v.data):.6f}   valido: {v.is_valid()}")

print(psi.draw("latex_source"))                      # codigo LaTeX para el cuaderno

## Experimento 2 · Regla de Born y número de disparos

Para `psi`, la regla de Born da $\Pr(0) = 5/9 \approx 0.556$ y $\Pr(1) = 4/9 \approx 0.444$.
Simulamos la medición con distinto número de disparos (*shots*) y vemos cómo la frecuencia se acerca a la probabilidad (ley de los grandes números).
Esto anticipa el error estadístico al estimar el kernel cuántico.

In [ ]:
# Probabilidades exactas (regla de Born)
print("Probabilidades exactas:", psi.probabilities_dict())

# Frecuencias con distinto numero de disparos
for shots in [100, 1000, 10000, 100000]:
    cuentas = psi.sample_counts(shots)
    f0 = cuentas.get("0", 0) / shots
    print(f"shots = {shots:6d}   frecuencia(0) = {f0:.4f}   error = {abs(f0 - 5/9):.4f}")

In [ ]:
# Histograma para el cuaderno (guardalo como figura)
fig = plot_histogram(psi.sample_counts(1000))
fig.savefig("histograma_psi.png", dpi=200, bbox_inches="tight")
fig

## Experimento 3 · Fase global

Tras medir, Qiskit puede devolver $\frac{1+2i}{\sqrt 5}|0\rangle$ o $-|1\rangle$.
Comprobamos que son iguales a $|0\rangle$ y $|1\rangle$ **salvo fase global** con el método `equiv`.

In [ ]:
resultado, estado = psi.measure()
print("Resultado:", resultado)
print("Estado tras medir:", estado.draw("latex_source"))

cero, uno = Statevector.from_label("0"), Statevector.from_label("1")
esperado = cero if resultado == "0" else uno
print("Igual salvo fase global:", estado.equiv(esperado))
print("Igual como vector:      ", np.allclose(estado.data, esperado.data))

## Experimento 4 · Interferencia: distinguir $|+\rangle$ y $|-\rangle$

Medidos directamente, $|+\rangle$ y $|-\rangle$ dan las mismas probabilidades.
Aplicando antes $H$, se distinguen con certeza: $H|+\rangle = |0\rangle$ y $H|-\rangle = |1\rangle$.

In [ ]:
H = Operator.from_label("H")
mas   = Statevector.from_label("+")
menos = Statevector.from_label("-")

print("Medida directa   |+>:", mas.probabilities_dict())
print("Medida directa   |->:", menos.probabilities_dict())
print("Tras H           |+>:", mas.evolve(H).probabilities_dict())
print("Tras H           |->:", menos.evolve(H).probabilities_dict())

## Experimento 5 · Raíz cuadrada de NOT: $(HSH)^2 = X$

Construimos el circuito $H, S, H, H, S, H$ y comparamos su matriz con la de $X$.

In [ ]:
R = QuantumCircuit(1)
R.h(0); R.s(0); R.h(0)
print("Matriz de R = HSH:")
print(np.round(Operator.from_circuit(R).data, 4))

R2 = R.compose(R)                       # aplicar R dos veces
display(R2.draw(output="mpl"))
R2.draw(output="mpl").savefig("circuito_R2.png", dpi=200, bbox_inches="tight")

X = Operator.from_label("X")
print("R^2 == X:", Operator.from_circuit(R2).equiv(X))
print(np.round(Operator.from_circuit(R2).data, 10))

## Experimento 6 · Las matrices son unitarias

Comprobamos $U^\dagger U = I$ para las puertas del cuaderno.

In [ ]:
for nombre in ["X", "Y", "Z", "H", "S", "T"]:
    U = Operator.from_label(nombre).data
    print(f"{nombre}:  U^dagger U = I ?  {np.allclose(U.conj().T @ U, np.eye(2))}")